# Binding knockout — instruct models

Reproduces the **instruct-side** rows of Tables 1, 2, 4, 5, 7, 8, 10, 12 and 17
of the paper: baseline binding (S) and knowledge (K) scores, edge knockout
R→item and U→item (control) on both tasks, and the directional
differentiation P(R | a or b).

Rebuilt from `notebooks_fanout/pipeline_instruct_mistral.ipynb`, cells 1-9. Shared helpers now live in `common/` (config, text parsers,
factorial-pair construction, chat formatting, edge-knockout hooks, results
store, clustered stats, published targets).

Run top-to-bottom with `MODEL_KEY` set to one of `{mistral, llama, gemma2,
nemo}`. Results pickles are written to `./results/<MODEL_KEY>_instruct/`;
the final validation gate reloads the stage-4 pickle and checks it against
the published targets.


In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}


In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
HF_TOKEN = config.HF_TOKEN          # read from the HF_TOKEN env var
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
ACTIVE_MODEL = config.ACTIVE_MODEL

from common.text_parsers import extract_options
from common.instruct.data import (load_n4, build_factorial_as_conditions,
                                  build_knowledge_probes)
from common.instruct.prompts import (format_for_chat, find_option_token_ids,
                                     detect_spans)
from common.instruct.hooks import edge_knockout, compute_logit_scores_edge
from common.instruct.store import init_results_store, save_results
from common.stats_utils import ttest_clustered, cond_p_R


In [ ]:
# ============================================================
# INSTRUCT FAN-OUT SETUP - GPU cleanup + N4 presence check
# ============================================================
import os, gc
import torch

_to_free = ["model", "tokenizer",
            "scores_base", "scores_ko", "scores_ctrl",
            "kscores_base", "kscores_ko", "kscores_ctrl",
            "lps_ko_R", "lps_ko_U",
            "texts_fmt", "texts_fmt_k",
            "positions", "positions_k",
            "results_sscore", "results_kscore"]
for _n in _to_free:
    if _n in globals():
        del globals()[_n]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  free {free/1e9:.1f} / {total/1e9:.1f} GB")
else:
    print("CUDA not available")

_runtime_n4 = os.path.join(DATA_DIR, "N4_1k.pkl")
if os.path.exists(_runtime_n4):
    print(f"[OK] {_runtime_n4}  (already present)")
else:
    print(f"!! MISSING N4_1k.pkl at {_runtime_n4}")


In [ ]:
import os
import re
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager
from itertools import combinations

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy import stats
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)


In [ ]:
print("=" * 80)
print(f"STAGE 1: S-SCORES — {CFG['label']}")
print("=" * 80)

# ── Load data ──
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# ── Load model ──
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
config.model = model; config.tokenizer = tokenizer

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
config.first_device = first_device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# ── Format texts ──
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# ── Compute S-scores ──
print(f"\n  Computing S-scores...")
results_sscore = {}
for cond in conditions:
    scores, c_chosen, p_c_list = [], [], []
    for i, text in enumerate(texts_fmt[cond]):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
        scores.append(S)
        chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
        c_chosen.append(1 if chosen == 'c' else 0)
        lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
        p_c_list.append(np.exp(lp_c - lp_all))
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            print(f"    {cond}: {i}/{n_total}")
            torch.cuda.empty_cache()
    results_sscore[cond] = {
        'S': np.array(scores), 'c_chosen': np.array(c_chosen),
        'p_c': np.array(p_c_list),
    }

# ── Print results ──
print(f"\n{'':20s}  {'Mean S':>8s}  {'Med S':>8s}  {'(c) rate':>8s}  {'Mean P(c)':>9s}")
for cond in conditions:
    r = results_sscore[cond]
    print(f"  {cond:20s}  {r['S'].mean():8.3f}  {np.median(r['S']):8.3f}  "
          f"{r['c_chosen'].mean():8.3f}  {r['p_c'].mean():9.3f}")
delta_S = results_sscore['B_cult']['S'].mean() - results_sscore['B_unrel']['S'].mean()
print(f"\n  Δ(S) = {delta_S:.4f}")

In [ ]:
# ================================================================
# KNOWLEDGE EVALUATION — STANDALONE
# Requires: imports + config + shared helpers + edge knockout, model loaded.
# ================================================================
import pickle
import numpy as np
from scipy.stats import ttest_rel

FINAL_HEADS = CFG['heads']
SAVE_DIR = Path(OUTPUT_DIR)  # curation: historical run wrote into the Drive data dir; results now go under results/

print("=" * 80)
print(f"KNOWLEDGE EVALUATION — {CFG['label']}")
print(f"  Heads: {FINAL_HEADS}")
print("=" * 80)

# ── 1. Load data ──
print("\n  Loading data...")
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
conditions = ['B_cult', 'B_unrel']
print(f"  {n_total} factorial pairs")

# ── 2. Build knowledge probes ──
knowledge = build_knowledge_probes(data)
k_conditions = ['K_cult', 'K_unrel']

# ── 3. Init store ──
store = init_results_store()
store['factorial'] = {
    'n_pairs': n_total,
    'items': list(data['items_cult']),
    'assoc_pos': list(data['assoc_pos']),
    'correct_group': list(data['correct_group']),
    'wrong_group': list(data['wrong_group']),
    'third_group': list(data['third_group']),
}
store['heads'] = {str(k): v for k, v in FINAL_HEADS.items()}

# ── 4. Option tokens ──
option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

# ── 5. Format texts ──
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}
texts_fmt_k = {c: format_for_chat(knowledge[c], tokenizer) for c in k_conditions}


# ================================================================
# STAGE 1: S-SCORES + K-SCORES
# ================================================================
print(f"\n{'='*60}")
print("STAGE 1: S-SCORES + K-SCORES")
print(f"{'='*60}")

# ── S-scores (binding) ──
print("\n  Computing S-scores (binding)...")
results_sscore = {}
for cond in conditions:
    scores, c_chosen, p_c_list, lps = [], [], [], []
    for i, text in enumerate(texts_fmt[cond]):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
        scores.append(S)
        chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
        c_chosen.append(1 if chosen == 'c' else 0)
        lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
        p_c_list.append(np.exp(lp_c - lp_all))
        lps.append({'a': lp_a, 'b': lp_b, 'c': lp_c})
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            print(f"    {cond}: {i}/{n_total}")
            torch.cuda.empty_cache()
    results_sscore[cond] = {
        'S': np.array(scores), 'c_chosen': np.array(c_chosen),
        'p_c': np.array(p_c_list), 'logprobs': lps,
    }

delta_S = results_sscore['B_cult']['S'].mean() - results_sscore['B_unrel']['S'].mean()
print(f"\n  BINDING:  delta(S) = {delta_S:.4f}")

store['binding'] = {
    'S_match': results_sscore['B_cult']['S'].tolist(),
    'S_mismatch': results_sscore['B_unrel']['S'].tolist(),
    'delta_S_per_pair': (results_sscore['B_cult']['S'] - results_sscore['B_unrel']['S']).tolist(),
    'delta_S_mean': float(delta_S),
    'logprobs_match': results_sscore['B_cult']['logprobs'],
    'logprobs_mismatch': results_sscore['B_unrel']['logprobs'],
}

# ── K-scores (knowledge) ──
print("\n  Computing K-scores (knowledge)...")
results_kscore = {}
for cond in k_conditions:
    scores, c_chosen, p_c_list, lps = [], [], [], []
    for i, text in enumerate(texts_fmt_k[cond]):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        K = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
        scores.append(K)
        chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
        c_chosen.append(1 if chosen == 'c' else 0)
        lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
        p_c_list.append(np.exp(lp_c - lp_all))
        lps.append({'a': lp_a, 'b': lp_b, 'c': lp_c})
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            print(f"    {cond}: {i}/{n_total}")
            torch.cuda.empty_cache()
    results_kscore[cond] = {
        'K': np.array(scores), 'c_chosen': np.array(c_chosen),
        'p_c': np.array(p_c_list), 'logprobs': lps,
    }

delta_K = results_kscore['K_cult']['K'].mean() - results_kscore['K_unrel']['K'].mean()
print(f"  KNOWLEDGE: delta(K) = {delta_K:.4f}")
if abs(delta_S) > 1e-6:
    print(f"  Ratio |dK/dS| = {abs(delta_K/delta_S):.2f}")

store['knowledge'] = {
    'K_match': results_kscore['K_cult']['K'].tolist(),
    'K_mismatch': results_kscore['K_unrel']['K'].tolist(),
    'delta_K_per_pair': (results_kscore['K_cult']['K'] - results_kscore['K_unrel']['K']).tolist(),
    'delta_K_mean': float(delta_K),
    'logprobs_match': results_kscore['K_cult']['logprobs'],
    'logprobs_mismatch': results_kscore['K_unrel']['logprobs'],
}

save_results(store, suffix="_stage1")


# ================================================================
# STAGE 4: KNOCKOUT ON BOTH TASKS
# ================================================================
print(f"\n{'='*60}")
print("STAGE 4: KNOCKOUT DISSOCIATION")
print(f"{'='*60}")

# ── Build positions for binding ──
print("\n  Building positions (binding)...")
positions = {c: [] for c in conditions}
for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens,
            })
        del enc

# ── Build positions for knowledge ──
print("  Building positions (knowledge)...")
positions_k = {c: [] for c in k_conditions}
for c in k_conditions:
    for i in range(n_total):
        q_text = knowledge[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt_k[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions_k[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions_k[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens,
            })
        del enc


# ── A. BINDING knockout ──
print("\n  [BINDING] Baseline...")
scores_base = {}
for cond in conditions:
    scores_base[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        {}, positions[cond], 'B_to_item', option_tokens)
delta_base = scores_base['B_cult'].mean() - scores_base['B_unrel'].mean()
diffs_base = scores_base['B_cult'] - scores_base['B_unrel']

print("  [BINDING] B->item KO...")
scores_ko = {}; lps_ko_R = {}
for cond in conditions:
    scores_ko[cond], lps_ko_R[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        FINAL_HEADS, positions[cond], 'B_to_item', option_tokens, return_lps=True)
delta_ko = scores_ko['B_cult'].mean() - scores_ko['B_unrel'].mean()
diffs_ko = scores_ko['B_cult'] - scores_ko['B_unrel']
items_arr = np.array(data['items_cult'])
t, p = ttest_clustered(diffs_base, diffs_ko, items_arr)
s_red = (1 - delta_ko / delta_base) * 100 if abs(delta_base) > 1e-10 else 0
print(f"  B->item KO: reduction={s_red:.1f}%, t={t:.3f}, p={p:.6f}")

print("  [BINDING] A->item KO (control)...")
scores_ctrl = {}; lps_ko_U = {}
for cond in conditions:
    scores_ctrl[cond], lps_ko_U[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        FINAL_HEADS, positions[cond], 'A_to_item', option_tokens, return_lps=True)
delta_ctrl = scores_ctrl['B_cult'].mean() - scores_ctrl['B_unrel'].mean()
diffs_ctrl = scores_ctrl['B_cult'] - scores_ctrl['B_unrel']
t_c, p_c = ttest_clustered(diffs_base, diffs_ctrl, items_arr)

store['knockout'] = {
    'binding': {
        'delta_baseline': float(delta_base),
        'delta_B_ko': float(delta_ko),
        'delta_A_ko': float(delta_ctrl),
        'reduction_B_pct': float(s_red),
        'reduction_A_pct': float((1 - delta_ctrl / delta_base) * 100) if abs(delta_base) > 1e-10 else 0,
        't_B': float(t), 'p_B': float(p),
        't_A': float(t_c), 'p_A': float(p_c),
        'diffs_base': diffs_base.tolist(),
        'diffs_B_ko': diffs_ko.tolist(),
        'diffs_A_ko': diffs_ctrl.tolist(),
    }
}


# ── B. KNOWLEDGE knockout ──
print("\n  [KNOWLEDGE] Baseline...")
kscores_base = {}
for cond in k_conditions:
    kscores_base[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[cond], knowledge[cond],
        {}, positions_k[cond], 'B_to_item', option_tokens)
delta_k_base = kscores_base['K_cult'].mean() - kscores_base['K_unrel'].mean()
kdiffs_base = kscores_base['K_cult'] - kscores_base['K_unrel']

print("  [KNOWLEDGE] B->item KO...")
kscores_ko = {}
for cond in k_conditions:
    kscores_ko[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[cond], knowledge[cond],
        FINAL_HEADS, positions_k[cond], 'B_to_item', option_tokens)
delta_k_ko = kscores_ko['K_cult'].mean() - kscores_ko['K_unrel'].mean()
kdiffs_ko = kscores_ko['K_cult'] - kscores_ko['K_unrel']
t_k, p_k = ttest_clustered(kdiffs_base, kdiffs_ko, items_arr)
k_red = (1 - delta_k_ko / delta_k_base) * 100 if abs(delta_k_base) > 1e-10 else 0
print(f"  B->item KO: reduction={k_red:.1f}%, t={t_k:.3f}, p={p_k:.6f}")

print("  [KNOWLEDGE] A->item KO (control)...")
kscores_ctrl = {}
for cond in k_conditions:
    kscores_ctrl[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[cond], knowledge[cond],
        FINAL_HEADS, positions_k[cond], 'A_to_item', option_tokens)
delta_k_ctrl = kscores_ctrl['K_cult'].mean() - kscores_ctrl['K_unrel'].mean()
kdiffs_ctrl = kscores_ctrl['K_cult'] - kscores_ctrl['K_unrel']
t_k_c, p_k_c = ttest_clustered(kdiffs_base, kdiffs_ctrl, items_arr)
k_ctrl_red = (1 - delta_k_ctrl / delta_k_base) * 100 if abs(delta_k_base) > 1e-10 else 0

store['knockout']['knowledge'] = {
    'delta_baseline': float(delta_k_base),
    'delta_B_ko': float(delta_k_ko),
    'delta_A_ko': float(delta_k_ctrl),
    'reduction_B_pct': float(k_red),
    'reduction_A_pct': float(k_ctrl_red),
    't_B': float(t_k), 'p_B': float(p_k),
    't_A': float(t_k_c), 'p_A': float(p_k_c),
    'diffs_base': kdiffs_base.tolist(),
    'diffs_B_ko': kdiffs_ko.tolist(),
    'diffs_A_ko': kdiffs_ctrl.tolist(),
}


# ── DISSOCIATION SUMMARY ──
print(f"\n  {'='*55}")
print(f"  DISSOCIATION: BINDING vs KNOWLEDGE under B->item KO")
print(f"  {'='*55}")
print(f"  |dS| baseline = {abs(delta_base):.4f}")
print(f"  |dK| baseline = {abs(delta_k_base):.4f}")
print(f"  Ratio |dK/dS| = {abs(delta_k_base/delta_base):.2f}")
print(f"  {'---'*17}")
print(f"  Binding  |dS| B->item: {s_red:+.1f}%  (p={p:.4f})")
print(f"  Binding  |dS| A->item: {(1-delta_ctrl/delta_base)*100:+.1f}%  (p={p_c:.4f})  [control]")
print(f"  Knowledge|dK| B->item: {k_red:+.1f}%  (p={p_k:.4f})")
print(f"  Knowledge|dK| A->item: {k_ctrl_red:+.1f}%  (p={p_k_c:.4f})  [control]")
print(f"  {'---'*17}")
ks_ratio = abs(k_red / s_red) if abs(s_red) > 0.1 else float('nan')
print(f"  K/S ratio = {ks_ratio:.2f}")
if ks_ratio < 0.5:
    print(f"  -> K < S: heads mediate GATING more than knowledge")
elif ks_ratio > 1.5:
    print(f"  -> K > S: heads mediate KNOWLEDGE more than gating")
else:
    print(f"  -> K ~ S: heads participate equally in both")
print(f"  {'='*55}")


# ── Significance of baseline pairing effects ──
from scipy.stats import ttest_1samp

items_arr = np.array(data['items_cult'])
unique_items = np.unique(items_arr)

dS_per_pair = results_sscore['B_cult']['S'] - results_sscore['B_unrel']['S']
dK_per_pair = results_kscore['K_cult']['K'] - results_kscore['K_unrel']['K']

# Cluster to item-level means (n=66)
dS_item = np.array([dS_per_pair[items_arr == it].mean() for it in unique_items])
dK_item = np.array([dK_per_pair[items_arr == it].mean() for it in unique_items])

t_dS, p_dS = ttest_1samp(dS_item, 0)
t_dK, p_dK = ttest_1samp(dK_item, 0)

print(f"\n  Significance (cluster-corrected, n={len(unique_items)} items):")
print(f"    Δ(S) ≠ 0: t={t_dS:.3f}, p={p_dS:.2e}")
print(f"    Δ(K) ≠ 0: t={t_dK:.3f}, p={p_dK:.2e}")

store['binding']['t_baseline'] = float(t_dS)
store['binding']['p_baseline'] = float(p_dS)
store['knowledge']['t_baseline'] = float(t_dK)
store['knowledge']['p_baseline'] = float(p_dK)


# ── Directional differentiation: P(R | a or b) ──
assoc_pos_arr = list(data['assoc_pos'])

pR_match = cond_p_R(results_sscore['B_cult']['logprobs'],  assoc_pos_arr)
pR_mism  = cond_p_R(results_sscore['B_unrel']['logprobs'], assoc_pos_arr)

pR_match_item = np.array([pR_match[items_arr == it].mean() for it in unique_items])
pR_mism_item  = np.array([pR_mism[items_arr == it].mean()  for it in unique_items])

t_dir, p_dir = ttest_1samp(pR_match_item, 0.5)

print(f"\n  Directional differentiation P(R | a or b):")
print(f"    Match    : mean = {pR_match_item.mean():.3f}  (t vs 0.5: t={t_dir:.2f}, p={p_dir:.2e})")
print(f"    Mismatch : mean = {pR_mism_item.mean():.3f}  [sanity ~0.5]")

store['binding']['pR_match_per_pair'] = pR_match.tolist()
store['binding']['pR_mism_per_pair']  = pR_mism.tolist()
store['binding']['pR_match_mean'] = float(pR_match_item.mean())
store['binding']['pR_mism_mean']  = float(pR_mism_item.mean())
store['binding']['t_directional'] = float(t_dir)
store['binding']['p_directional'] = float(p_dir)


store['binding']['logprobs_match_Rko']    = lps_ko_R['B_cult']
store['binding']['logprobs_mismatch_Rko'] = lps_ko_R['B_unrel']
store['binding']['logprobs_match_Uko']    = lps_ko_U['B_cult']
store['binding']['logprobs_mismatch_Uko'] = lps_ko_U['B_unrel']

save_results(store, suffix="_stage4_ko")
print("\n  Done.")

In [ ]:
# ================================================================
# VALIDATION GATE - reproduce published instruct row + integrity
# ================================================================
import pickle
from pathlib import Path

from common.published_targets import TARGETS_INSTRUCT
TARGETS = TARGETS_INSTRUCT

ABS_TOL = 0.05   # instruct values are ~1-12, two-decimal rounding
PCT_TOL = 1.0    # percentage points

_pkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_stage4_ko.pkl"
print(f"  Reloading: {_pkl}")
with open(_pkl, "rb") as f:
    _store = pickle.load(f)

T = TARGETS.get(ACTIVE_MODEL)
assert T is not None, f"No TARGETS entry for {ACTIVE_MODEL}"

_ok = True
def _check(cond, msg):
    global _ok
    mark = "OK  " if cond else "FAIL"
    print(f"  [{mark}] {msg}")
    if not cond:
        _ok = False

# 1. 8 logprob arrays integrity (baseline + R-KO + U-KO)
EXPECTED_LP_KEYS = [
    ("binding",   "logprobs_match"),
    ("binding",   "logprobs_mismatch"),
    ("knowledge", "logprobs_match"),
    ("knowledge", "logprobs_mismatch"),
    ("binding",   "logprobs_match_Rko"),
    ("binding",   "logprobs_mismatch_Rko"),
    ("binding",   "logprobs_match_Uko"),
    ("binding",   "logprobs_mismatch_Uko"),
]
for sec, key in EXPECTED_LP_KEYS:
    arr = _store.get(sec, {}).get(key)
    _check(arr is not None, f"{sec}.{key} present")
    if arr is not None:
        _check(len(arr) == 847, f"{sec}.{key} length == 847 (got {len(arr)})")
        sample = arr[0] if len(arr) else None
        _check(isinstance(sample, dict) and set(sample.keys()) == {"a", "b", "c"},
               f"{sec}.{key}[0] is dict with keys (a,b,c)")

# 2. diffs arrays integrity
for sec in ("binding", "knowledge"):
    for key in ("diffs_base", "diffs_B_ko", "diffs_A_ko"):
        arr = _store.get("knockout", {}).get(sec, {}).get(key)
        _check(arr is not None, f"knockout.{sec}.{key} present")
        if arr is not None:
            _check(len(arr) == 847, f"knockout.{sec}.{key} length == 847 (got {len(arr)})")

# 3. Factorial integrity
_assoc = set(_store["factorial"]["assoc_pos"])
_check(_assoc.issubset({"a", "b"}), f"factorial.assoc_pos subset of (a,b)  (got {_assoc})")
_items_unique = len(set(_store["factorial"]["items"]))
_check(_items_unique == 66, f"unique factorial.items == 66 (got {_items_unique})")

# 4. Reproduction
b = _store["knockout"]["binding"]
k = _store["knockout"]["knowledge"]
abs_dS = abs(b["delta_baseline"])
abs_dK = abs(k["delta_baseline"])
red_B_S = b["reduction_B_pct"]
red_A_S = b["reduction_A_pct"]
red_B_K = k["reduction_B_pct"]
red_A_K = k["reduction_A_pct"]
KS_ratio = red_B_K / red_B_S if abs(red_B_S) > 1e-6 else float("nan")

def _within(val, target, tol, label):
    if target is None:
        print(f"  [SKIP] {label} (no published target)")
        return
    dev = abs(val - target)
    cond = dev < tol
    _check(cond, f"{label}: target={target:+.4f}  got={val:+.4f}  dev={dev:.4f}  tol={tol:.4f}")

_within(abs_dS, T["abs_dS"], ABS_TOL, "|dS| baseline")
_within(abs_dK, T["abs_dK"], ABS_TOL, "|dK| baseline")
_within(red_B_S, T["red_B_S"], PCT_TOL, "R->item dS reduction (pp)")
_within(red_B_K, T["red_B_K"], PCT_TOL, "R->item dK reduction (pp)")
_within(red_A_S, T["red_A_S"], PCT_TOL, "U->item dS reduction (pp)")
_within(red_A_K, T["red_A_K"], PCT_TOL, "U->item dK reduction (pp)")

print(f"\n  Reported metrics:")
print(f"    |dS| baseline      = {abs_dS:.4f}   (target {T['abs_dS']})")
print(f"    |dK| baseline      = {abs_dK:.4f}   (target {T['abs_dK']})")
print(f"    R->item dS red.    = {red_B_S:+.1f}%  (target {T['red_B_S']})")
print(f"    R->item dK red.    = {red_B_K:+.1f}%  (target {T['red_B_K']})")
print(f"    U->item dS red.    = {red_A_S:+.1f}%  (target {T['red_A_S']})  [control]")
print(f"    U->item dK red.    = {red_A_K:+.1f}%  (target {T['red_A_K']})  [control]")
print(f"    K/S ratio (R-KO)   = {KS_ratio:.2f}   (target {T['KS_ratio']})")
print(f"    final path   = {_pkl}")

print(f"\n  === GATE [{ACTIVE_MODEL} instruct]: {'GREEN - faithful regen' if _ok else 'RED - investigate'} ===")
